# 머신러닝 기반 영화 리뷰 감성 분석
1. 데이터 준비 (전처리된 데이터 -> 특징 벡터 추출 (TF-IDF))
2. 머신러닝 모델별 학습 및 평가
3. 배포 준비 : 영화 리뷰 긍부정 판단 -> 클래스 생성, 모델 저장


## 1.데이터 준비 (전처리된 데이터)

In [1]:
import pandas as pd

data_filename = './data/Korean_movie_reviews_2016.csv'

# 데이터 로딩
review_df = pd.read_csv(data_filename)
review_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [3]:
# 입력 데이터와 정답 데이터 추출
review_list = list(review_df.review) 
label_list = list(review_df.label)

In [4]:
# 학습 데이터와 평가 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.2)
len(train_X), len(test_X), len(train_y), len(test_y)

(132307, 33077, 132307, 33077)

## 2. 특징 벡터 추출 : TF-IDF

In [52]:
# 한국어 감성 분석용 tokenizer 정의
from konlpy.tag import Okt

def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

In [53]:
# 최대 단어 수 1000개
# 학습 데이터로 Vectorizer 생성 및 학습 데이터 특징 벡터 추출
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=korean_tokenizer, max_features=1000)
vectorizer.fit(train_X)
len(vectorizer.get_feature_names_out())


c:\Users\user\anaconda3\envs\aiservice26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1000

In [8]:
# 테스트 데이터 특징 벡터 추출
train_X_fv = vectorizer.transform(train_X)
print(train_X_fv.shape)
print(train_X_fv)

(132307, 1000)
  (np.int32(0), np.int32(457))	0.4870254087934076
  (np.int32(0), np.int32(623))	0.18413382595319291
  (np.int32(0), np.int32(630))	0.6333168023035561
  (np.int32(0), np.int32(853))	0.5725476515098563
  (np.int32(1), np.int32(17))	0.19244421337700393
  (np.int32(1), np.int32(33))	0.2593627191036425
  (np.int32(1), np.int32(225))	0.2706568755552622
  (np.int32(1), np.int32(359))	0.18962782314114981
  (np.int32(1), np.int32(372))	0.1753459319799296
  (np.int32(1), np.int32(403))	0.23912456026502105
  (np.int32(1), np.int32(613))	0.17889102834484089
  (np.int32(1), np.int32(623))	0.09783323475259245
  (np.int32(1), np.int32(627))	0.3263300637857622
  (np.int32(1), np.int32(687))	0.20298993734110093
  (np.int32(1), np.int32(695))	0.24957803414585797
  (np.int32(1), np.int32(804))	0.2939476244323716
  (np.int32(1), np.int32(820))	0.22334130693998613
  (np.int32(1), np.int32(895))	0.27674731766517313
  (np.int32(1), np.int32(966))	0.3143884684496709
  (np.int32(1), np.int32(96

In [9]:
train_X_fv.toarray()[:1]

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

In [10]:
# 테스트 데이터 특징 벡터 추출
test_X_fv = vectorizer.transform(test_X)
print(test_X_fv)

  (np.int32(0), np.int32(24))	0.4659976562343131
  (np.int32(0), np.int32(360))	0.4817276671317421
  (np.int32(0), np.int32(514))	0.4469099534052267
  (np.int32(0), np.int32(564))	0.3768958911625621
  (np.int32(0), np.int32(608))	0.4318115870373671
  (np.int32(0), np.int32(623))	0.1501478377344242
  (np.int32(1), np.int32(91))	0.2781341394923802
  (np.int32(1), np.int32(391))	0.17386119181743295
  (np.int32(1), np.int32(457))	0.2170591921580285
  (np.int32(1), np.int32(520))	0.17423396579443073
  (np.int32(1), np.int32(564))	0.20599774220583192
  (np.int32(1), np.int32(584))	0.19847577791152207
  (np.int32(1), np.int32(585))	0.19170959009628954
  (np.int32(1), np.int32(623))	0.16413081859275708
  (np.int32(1), np.int32(653))	0.28794534652311854
  (np.int32(1), np.int32(718))	0.16931717860973006
  (np.int32(1), np.int32(719))	0.21536342493353097
  (np.int32(1), np.int32(754))	0.17239565770937237
  (np.int32(1), np.int32(789))	0.3276602541981567
  (np.int32(1), np.int32(836))	0.284927569

In [11]:
# 정답 데이터 변환 (np.array)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)

In [14]:
# 데이터 일부 확인
print(train_y[:10])
print(test_y[:10])

[0 1 1 0 0 1 0 1 0 0]
[1 0 1 1 0 0 0 0 1 1]


## 3. 머신러닝 모델별 학습 및 평가
* 의사결정 트리
* 랜덤포레스트
* 나이브 베이즈
* 로지스틱 회귀 분석
* SVM
* Perceptron

In [21]:
# 머신러닝 모델별 학습 성능 평가 결과 저장 준비
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])

## 3.1 의사결정 트리 (Decision Tree)

In [16]:
# 학습
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [19]:
# 성능 평가
train_score = dtc.score(train_X_fv, train_y) * 100
test_score = dtc.score(test_X_fv, test_y) * 100
train_score, test_score

(98.16033921107726, 79.81376787495843)

In [22]:
# 평가 결과 score_df에 추가
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.160339,79.813768


## 3.2 랜덤 포레스트 (Random Forrest)

In [23]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs = -1)
rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [24]:
train_score = rf.score(train_X_fv, train_y) * 100
test_score = rf.score(test_X_fv, test_y) * 100
print(train_score, test_score)

98.15882757526056 84.74770988904677


In [25]:
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.160339,79.813768
RandomForest,98.158828,84.747710


## 3.3 나이브 베이즈 (Naive Baysian)

In [26]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [27]:
train_score = mnb.score(train_X_fv, train_y) * 100
test_score = mnb.score(test_X_fv, test_y) * 100
print(train_score, test_score)

85.29556259306008 85.21026695286756


In [28]:
score_df.loc['NaiveBayes'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.160339,79.813768
RandomForest,98.158828,84.747710
NaiveBayes,85.295563,85.210267


## 3.4 로지스틱 회귀 분석 (Logistic Regression)

In [29]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [30]:
train_score = lr.score(train_X_fv, train_y) * 100
test_score = lr.score(test_X_fv, test_y) * 100
print(train_score, test_score)

86.21917207706319 85.96305589987


In [31]:
score_df.loc['LogisticRegression'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.160339,79.813768
RandomForest,98.158828,84.747710
NaiveBayes,85.295563,85.210267
LogisticRegression,86.219172,85.963056


## 3.5 SVM (Support Vector Machine)

In [32]:
from sklearn.svm import LinearSVC
svc = LinearSVC(verbose=True)
svc.fit(train_X_fv, train_y)

[LibLinear]

LinearSVC(verbose=True)

In [33]:
train_score = svc.score(train_X_fv, train_y) * 100
test_score = svc.score(test_X_fv, test_y) * 100
print(train_score, test_score)

86.20859062634631 85.89956767542401


In [34]:
score_df.loc['SVM'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.160339,79.813768
RandomForest,98.158828,84.747710
NaiveBayes,85.295563,85.210267
LogisticRegression,86.219172,85.963056
SVM,86.208591,85.899568


## 3.6 Perceptron

## 3.7 성능 비교

In [38]:
# 평가 결과 저장 데이터 프레임 확인
score_df.sort_values(by='test', ascending=False)

,train,test
LogisticRegression,86.219172,85.963056
SVM,86.208591,85.899568
NaiveBayes,85.295563,85.210267
RandomForest,98.158828,84.747710
DecisionTree,98.160339,79.813768


## 4. 영화 리뷰 긍부정 판단
* 학습된 모델 중 선택하여 활용

In [ ]:
# 영화 리뷰감성 분석용 tokenizer 정의
from konlpy.tag import Okt

def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

# 특징 벡터 추출 모델 : vectorizer

# 학습 모델 선택
sa_model = lr


In [ ]:
review = '영화가 재미있다'

# 텍스트 전처리

# 특징 벡터 추출
review_fv = vectorizer.transform([review])

# 예측
pred = sa_model.predict(review_fv)

# 예측 결과 출력
result = '긍정' if pred[0] >= 0.5 else '부정'
print(f'{review} -> {result}({pred[0]})')


영화가 재미있다 -> 긍정(1)


In [48]:
# 함수로 만들기
def analyze_semtiment(review):
    # 특징 백터 추출
    review_fv = vectorizer.transform([review])
    # 학습된 모델로 예측
    pred = sa_model.predict(review_fv)
    # 예측값에 따라 결과를 생성하여 변환
    result = '긍정' if pred[0] >= 0.5 else '부정'
    return result, pred[0]


In [49]:
# 함수 테스트
reviews = [
    '이 영화 개꿀잼 ㅋㅋㅋ',
    '하품만 나온다',
    '이 영화 핵노잼 ㅠㅠ',
    '이딴게 영화냐 ㅉㅉ',
    '와 개쩐다',
    '감독 뭐하는 놈이냐',
    '정말 세계관 최강자들의 영화다'
]

for review in reviews:
    sentiment, prob = analyze_semtiment(review)
    print(f'{review} -> {sentiment}({prob})')

이 영화 개꿀잼 ㅋㅋㅋ -> 부정(0)
하품만 나온다 -> 긍정(1)
이 영화 핵노잼 ㅠㅠ -> 부정(0)
이딴게 영화냐 ㅉㅉ -> 부정(0)
와 개쩐다 -> 긍정(1)
감독 뭐하는 놈이냐 -> 부정(0)
정말 세계관 최강자들의 영화다 -> 긍정(1)


In [50]:
# 문장을 입력 받아서 긍부정 판단 
review = input('>> 리뷰 입력 : ')
sentiment, prob = analyze_semtiment(review)
print(f'{review} -> {sentiment}({prob})')

영화를 보다가 졸았다. -> 부정(0)


In [54]:
# 배포를 위한 모델 저장

# 특징 추출용 vectorizer
import joblib
joblib.dump(vectorizer, './model/sa_movie_vectorizer.pkl')

# 감성 분석 모델 sa_model
joblib.dump(sa_model, './model/sa_movie_predict.pkl')

['./model/sa_movie_predict.pkl']

In [ ]:
import joblib
def korean_tokenizer(text):
        my_tags = ['Noun', 'Adjective', 'Verb']
        my_stopwords = []
        return [word for word, tag in Okt().pos(text) if word not in my_stopwords and tag in my_tags]

class SentimentAnalyzer:
    def __init__(self, tokenizer, vectorizer_file, predict_model_file):
        self.__vectorizer = joblib.loadload(vectorizer_file)
        self.__predict_model = joblib.load(predict_model_file)
        self.__tokenizer = tokenizer

    def analyze_semtiment(self, review):
    # 특징 백터 추출
        self.__review_fv = vectorizer.transform([review])
        # 학습된 모델로 예측
        self.__pred = sa_model.predict(review_fv)
        # 예측값에 따라 결과를 생성하여 변환
        self.__result = '긍정' if pred[0] >= 0.5 else '부정'
        return result, pred[0]
